In [24]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime, timedelta

standings_url =  'https://fbref.com/en/comps/31/schedule/Liga-MX-Scores-and-Fixtures'

In [16]:
data = requests.get(standings_url)
soup = BeautifulSoup(data.text)
standings_table = soup.select('table.stats_table')[0]


In [18]:
# Extraer los datos
matches = []
rows = soup.find_all("tr")  # Encontrar todas las filas de la tabla
for row in rows:
    home_team = row.find("td", {"data-stat": "home_team"})
    away_team = row.find("td", {"data-stat": "away_team"})
    date = row.find("td", {"data-stat": "date"})

In [30]:
# Extraer los datos
matches = []
rows = soup.find_all("tr")  # Encontrar todas las filas de la tabla

for row in rows:
    home_team = row.find("td", {"data-stat": "home_team"})
    away_team = row.find("td", {"data-stat": "away_team"})
    date = row.find("td", {"data-stat": "date"})
    time = row.find("td", {"data-stat": "start_time"})

    # Si la hora no está disponible, asignamos un NA
    if time is None or time.text.strip() == "NA":
        match_time = pd.NaT
    else:
        # Solo intentamos acceder al texto si time no es None
        match_time = time.find("span").text.strip() if time.find("span") else pd.NaT

    # Verificar si los datos de los equipos y la fecha están disponibles
    if home_team and away_team and date:
        matches.append({
            "Equipo Local": home_team.text.strip(),
            "Equipo Visitante": away_team.text.strip(),
            "Fecha": date.text.strip(),
            "Hora": match_time,
            "Local/Away": "Home" if home_team else "Away"  # Determinamos si es local o visitante
        })
# Convertir los datos a un DataFrame de pandas para generar la tabla
df = pd.DataFrame(matches)

# Convertir las fechas de string a formato datetime
df['Fecha'] = pd.to_datetime(df['Fecha'])

# Calcular el rango de fechas de la próxima semana
hoy = datetime.now()
una_semana_despues = hoy + timedelta(days=7)

# Filtrar los partidos de la próxima semana
df_proxima_semana = df[(df['Fecha'] > hoy) & (df['Fecha'] <= una_semana_despues)]
df_proxima_semana = df_proxima_semana.drop_duplicates()

# Mostrar la tabla filtrada
print(df_proxima_semana)

          Equipo Local Equipo Visitante      Fecha   Hora Local/Away
190  Atlético San Luis      Tigres UANL 2025-01-11  17:00       Home
191            Tijuana           Toluca 2025-01-10  19:00       Home
192          Querétaro          América 2025-01-10  19:00       Home
193           Mazatlán        FC Juárez 2025-01-10  20:00       Home
194         Pumas UNAM           Necaxa 2025-01-12  12:00       Home
195          Monterrey           Puebla 2025-01-11  19:00       Home
196        Guadalajara    Santos Laguna 2025-01-11  19:05       Home
197          Cruz Azul            Atlas 2025-01-11  21:00       Home
